## **ML Analysis**

##### Predict whether a customer will place another order within the next 90 days. - classification problem

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score)

df = pd.read_csv("customer_df.csv")
df = df.fillna(df.median(numeric_only=True))
df.head()

,customer_unique_id,total_orders,total_price,average_price,average_freight,average_review,average_installments,total_payment,average_photos,average_weight,approval_time,delivery_time,delivery_delay,shipping_delay,review_response,purchase_year,purchase_month,purchase_hour,Repeat_Purchase
0,0000366f3b9a7992bf8c76cfdf3221e2,1,129.90,129.90,12.00,5.0,8.0,141.90,1.0,1500.0,0.247500,6.0,-5.0,-4.0,4.0,2018,5,10.0,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,18.90,18.90,8.29,4.0,1.0,27.19,1.0,375.0,7.238056,3.0,-5.0,-3.0,0.0,2018,5,11.0,0
2,0000f46a3911fa3c0805444483337064,1,69.00,69.00,17.22,3.0,8.0,86.22,3.0,1500.0,0.000000,25.0,-2.0,-3.0,1.0,2017,3,21.0,0
3,0000f6ccb0745a6a4b88665a16c9f078,1,25.99,25.99,17.63,4.0,4.0,43.62,5.0,150.0,0.326667,20.0,-12.0,-6.0,1.0,2017,10,20.0,0
4,0004aac84e0df4da2b147fca70cf8255,1,180.00,180.00,16.89,5.0,6.0,196.89,3.0,6050.0,0.352778,13.0,-8.0,-7.0,4.0,2017,11,19.0,0


In [7]:
df.keys()

Index(['customer_unique_id', 'total_orders', 'total_price', 'average_price',
       'average_freight', 'average_review', 'average_installments',
       'total_payment', 'average_photos', 'average_weight', 'approval_time',
       'delivery_time', 'delivery_delay', 'shipping_delay', 'review_response',
       'purchase_year', 'purchase_month', 'purchase_hour', 'Repeat_Purchase'],
      dtype='str')

In [8]:
print("shape:",df.shape)
print("--------------------------------------------------")
print(df.isnull().sum())
print("--------------------------------------------------")
print("is any duplicated value:", df.duplicated().sum())
print("--------------------------------------------------")
print(df["Repeat_Purchase"].value_counts())

shape: (93335, 19)
--------------------------------------------------
customer_unique_id      0
total_orders            0
total_price             0
average_price           0
average_freight         0
average_review          0
average_installments    0
total_payment           0
average_photos          0
average_weight          0
approval_time           0
delivery_time           0
delivery_delay          0
shipping_delay          0
review_response         0
purchase_year           0
purchase_month          0
purchase_hour           0
Repeat_Purchase         0
dtype: int64
--------------------------------------------------
is any duplicated value: 0
--------------------------------------------------
Repeat_Purchase
0    91377
1     1958
Name: count, dtype: int64


### **Model prepration and Training**

In [9]:
X = df.drop(columns=["customer_unique_id", "Repeat_Purchase"])
y = df["Repeat_Purchase"]

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

Training data: (74668, 17)
Testing data: (18667, 17)
Training labels: (74668,)
Testing labels: (18667,)


In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [12]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42 ), 
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "XGBoost": XGBClassifier(eval_metric="logloss",random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5)}

results = []

for name, model in models.items():
    if name in ["Logistic Regression", "KNN"]:
        model.fit(X_train_scaled, y_train)
        train_pred = model.predict(X_train_scaled)
        test_pred = model.predict(X_test_scaled)
        train_prob = model.predict_proba(X_train_scaled)[:, 1]
        test_prob = model.predict_proba(X_test_scaled)[:, 1]

    else:
        model.fit(X_train, y_train)
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)
        train_prob = model.predict_proba(X_train_scaled)[:, 1]
        test_prob = model.predict_proba(X_test_scaled)[:, 1]

    results.append({
        "Model": name,

        "Train Accuracy": accuracy_score(y_train, train_pred),
        "Test Accuracy": accuracy_score(y_test, test_pred),

        "Train Precision": precision_score(y_train, train_pred),
        "Test Precision": precision_score(y_test, test_pred),

        "Train Recall": recall_score(y_train, train_pred),
        "Test Recall": recall_score(y_test, test_pred),

        "Train F1": f1_score(y_train, train_pred),
        "Test F1": f1_score(y_test, test_pred),

        "Train ROC-AUC": roc_auc_score(y_train, train_prob),
        "Test ROC-AUC": roc_auc_score(y_test, test_prob),})

results_df = pd.DataFrame(results)
results_df

f:\GUVI_Class\GUVI_mini_Projects\Final_Project\Project1\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
f:\GUVI_Class\GUVI_mini_Projects\Final_Project\Project1\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
f:\GUVI_Class\GUVI_mini_Projects\Final_Project\Project1\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
f:\GUVI_Class\GUVI_mini_Projects\Final_Project\Project1\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
f:\GUVI_Class\GUVI_mini_Projects\Final_Project\P

,Model,Train Accuracy,Test Accuracy,Train Precision,Test Precision,Train Recall,Test Recall,Train F1,Test F1,Train ROC-AUC,Test ROC-AUC
0,Logistic Regression,0.979375,0.980340,0.791667,0.800000,0.036030,0.031915,0.068924,0.061381,0.640019,0.623139
1,Decision Tree,1.000000,0.954358,1.000000,0.047529,1.000000,0.066489,1.000000,0.055432,0.499840,0.500574
2,Random Forest,0.999946,0.979590,1.000000,0.142857,0.997472,0.002660,0.998734,0.005222,0.528679,0.519421
3,Gradient Boosting,0.980433,0.979000,0.948148,0.289474,0.080910,0.029255,0.149097,0.053140,0.520475,0.527193
4,XGBoost,0.983112,0.979858,0.990826,0.500000,0.204804,0.055851,0.339445,0.100478,0.506199,0.476889
5,KNN,0.978893,0.979375,0.666667,0.090909,0.007585,0.002660,0.015000,0.005168,0.964513,0.518669


#### **Report:** over all my XGboost algorithm performs well in all the metrices when compared to other models

In [13]:
xgb_model = XGBClassifier(eval_metric="logloss",random_state=42)
xgb_model.fit(X_train, y_train)

train_predict = xgb_model.predict(X_train)
test_predict = xgb_model.predict(X_test)

In [14]:
import pickle
with open("xgb_model.pkl", "wb") as file:
    pickle.dump(xgb_model, file)

### Predict future revenue expected from each customer.- Regreesion Problem


### **Model Preparation**

In [19]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

In [20]:
revenue_df = pd.read_csv("revenue_dataset.csv")
revenue_df.head()

,customer_unique_id,total_orders,total_price,last_purchase_year,last_purchase_month,first_purchase_year,customer_tenure_days,average_installments,average_review,past_revenue,avg_order_value,future_revenue
0,0000f46a3911fa3c0805444483337064,1,69.00,2017,3,2017,0,8.0,3.0,86.22,86.22,0.0
1,0000f6ccb0745a6a4b88665a16c9f078,1,25.99,2017,10,2017,0,4.0,4.0,43.62,43.62,0.0
2,0004aac84e0df4da2b147fca70cf8255,1,180.00,2017,11,2017,0,6.0,5.0,196.89,196.89,0.0
3,0004bd2a26a76fe21f786e4fbd80607f,1,154.00,2018,4,2018,0,8.0,4.0,166.98,166.98,0.0
4,00053a61a98854899e70ed204dd4bafe,1,382.00,2018,2,2018,0,3.0,1.0,419.18,419.18,0.0


In [21]:
revenue_df.keys()

Index(['customer_unique_id', 'total_orders', 'total_price',
       'last_purchase_year', 'last_purchase_month', 'first_purchase_year',
       'customer_tenure_days', 'average_installments', 'average_review',
       'past_revenue', 'avg_order_value', 'future_revenue'],
      dtype='str')

In [22]:
# Check target variable (future_revenue is mostly 0 - this affects R2)
print("future_revenue = 0:", (revenue_df["future_revenue"] == 0).sum(), "out of", len(revenue_df))
print("Mean future_revenue:", round(revenue_df["future_revenue"].mean(), 2))
print("Correlation past_revenue vs future_revenue:", round(revenue_df["past_revenue"].corr(revenue_df["future_revenue"]), 3))

feature_cols = [
    "total_orders",
    "last_purchase_year",
    "last_purchase_month",
    "first_purchase_year",
    "customer_tenure_days",
    "average_installments",
    "average_review",
    "past_revenue",
    "avg_order_value",
]

X = revenue_df[feature_cols]
y = revenue_df["future_revenue"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

future_revenue = 0: 64707 out of 65277
Mean future_revenue: 1.37
Correlation past_revenue vs future_revenue: 0.011


In [23]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(random_state=42),
    "Gradient Boosting Regressor": GradientBoostingRegressor(random_state=42),
    "XGBoost Regressor": XGBRegressor(random_state=42),
    "LightGBM Regressor": LGBMRegressor(random_state=42),
    "CatBoost Regressor": CatBoostRegressor(random_state=42)}

In [24]:
results = []

for name, model in models.items():
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    # RMSE = square root of MSE (do NOT use mean_squared_error directly as RMSE)
    train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

    results.append({
        "Model": name,
        "Train RMSE": train_rmse,
        "Test RMSE": test_rmse,
        "Train MAE": mean_absolute_error(y_train, train_pred),
        "Test MAE": mean_absolute_error(y_test, test_pred),
        "Train R2": r2_score(y_train, train_pred),
        "Test R2": r2_score(y_test, test_pred),
    })

results_df = pd.DataFrame(results)
display(results_df)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001406 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 857
[LightGBM] [Info] Number of data points in the train set: 52221, number of used features: 9
[LightGBM] [Info] Start training from score 1.319349
Learning rate set to 0.076489
0:	learn: 20.8057501	total: 137ms	remaining: 2m 16s
1:	learn: 20.7723487	total: 144ms	remaining: 1m 11s
2:	learn: 20.7390703	total: 155ms	remaining: 51.5s
3:	learn: 20.7081641	total: 164ms	remaining: 40.9s
4:	learn: 20.6781755	total: 174ms	remaining: 34.7s
5:	learn: 20.6494750	total: 183ms	remaining: 30.4s
6:	learn: 20.6206687	total: 190ms	remaining: 26.9s
7:	learn: 20.5940045	total: 198ms	remaining: 24.6s
8:	learn: 20.5627152	total: 209ms	remaining: 23s
9:	learn: 20.5381619	total: 219ms	remaining: 21.6s
10:	learn: 20.5144042	total: 227ms	remaining: 20.5s
11:	

,Model,Train RMSE,Test RMSE,Train MAE,Test MAE,Train R2,Test R2
0,Linear Regression,20.806617,23.683839,2.607409,2.867283,0.002574,0.006458
1,Random Forest Regressor,9.399606,24.588482,1.093443,2.868061,0.796438,-0.070891
2,Gradient Boosting Regressor,19.110037,24.222807,2.470563,2.913296,0.158603,-0.039276
3,XGBoost Regressor,15.379991,25.740508,2.307391,3.254575,0.455008,-0.173589
4,LightGBM Regressor,19.598649,23.828999,2.567717,2.985362,0.115027,-0.005758
5,CatBoost Regressor,16.387372,23.747610,2.359809,2.890260,0.381277,0.001100


### **Report** Over all Linear Regression Performed well while considering the Test R2, Test RMSE and Test MAE


In [25]:
from sklearn.linear_model import LinearRegression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
train_predict = lr_model.predict(X_train)
test_predict = lr_model.predict(X_test)

In [26]:
import pickle
with open("lr_model.pkl", "wb") as file:
    pickle.dump(lr_model, file)